In [ ]:
import torch
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor, WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
)
import librosa
import numpy as np
import re
from typing import List, Dict, Tuple, Optional
import json
import soundfile as sf
import tempfile
import os
from difflib import SequenceMatcher
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')

In [ ]:
class ModelLoader:
    """Singleton class to load models once and reuse them"""
    _instance = None
    _models_loaded = False

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(ModelLoader, cls).__new__(cls)
        return cls._instance

    def load_models(self):
        if self._models_loaded:
            print("Models already loaded!")
            return

        print("Loading models... This may take a few minutes on first run.")

        # Pronunciation assessment models
        print("Loading Whisper model...")
        self.whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
        self.whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")

        print("Loading Wav2Vec2 model...")
        self.wav2vec_processor = Wav2Vec2Processor.from_pretrained("kresnik/wav2vec2-large-xlsr-korean")
        self.wav2vec_model = Wav2Vec2ForCTC.from_pretrained("kresnik/wav2vec2-large-xlsr-korean")

        # Translation models
        print("Loading translation models...")
        try:
            self.translation_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
            self.translation_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
            self.translation_available = True
            self.nllb_model = True
            print("✓ NLLB translation model loaded")
        except Exception as e:
            print(f"Failed to load NLLB, trying Helsinki-NLP: {e}")
            try:
                self.translation_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
                self.translation_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
                self.translation_available = True
                self.nllb_model = False
                print("✓ Helsinki-NLP translation model loaded")
            except Exception as e2:
                print(f"✗ Could not load translation models: {e2}")
                self.translation_available = False

        # Text analysis models
        print("Loading text analysis models...")
        try:
            self.summarizer = pipeline(
                "summarization",
                model="facebook/bart-large-cnn",
                device=0 if torch.cuda.is_available() else -1
            )
            self.summarization_available = True
            print("✓ BART summarization model loaded")
        except Exception as e:
            print(f"Failed to load BART, trying T5: {e}")
            try:
                self.summarizer = pipeline(
                    "summarization",
                    model="t5-small",
                    device=0 if torch.cuda.is_available() else -1
                )
                self.summarization_available = True
                print("✓ T5 summarization model loaded")
            except Exception as e2:
                print(f"✗ Could not load summarization model: {e2}")
                self.summarization_available = False

        try:
            self.ner_pipeline = pipeline(
                "ner",
                model="dbmdz/bert-large-cased-finetuned-conll03-english",
                aggregation_strategy="simple",
                device=0 if torch.cuda.is_available() else -1
            )
            self.ner_available = True
            print("✓ NER model loaded")
        except Exception as e:
            print(f"✗ Could not load NER model: {e}")
            self.ner_available = False

        # Move models to GPU if available
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Moving models to {self.device}...")

        self.whisper_model.to(self.device)
        self.wav2vec_model.to(self.device)

        if self.translation_available:
            self.translation_model.to(self.device)

        self._models_loaded = True
        print(f"✅ All models loaded successfully on {self.device}!")

# Load models once
model_loader = ModelLoader()
model_loader.load_models()

Loading models... This may take a few minutes on first run.
Loading Whisper model...


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Device set to use cuda:0


✓ NER model loaded
Moving models to cuda...
✅ All models loaded successfully on cuda!


In [ ]:
class KoreanPronunciationAssessor:
    def __init__(self, model_loader):
        """Initialize with pre-loaded models"""
        self.whisper_processor = model_loader.whisper_processor
        self.whisper_model = model_loader.whisper_model
        self.wav2vec_processor = model_loader.wav2vec_processor
        self.wav2vec_model = model_loader.wav2vec_model

        self.translation_model = getattr(model_loader, 'translation_model', None)
        self.translation_tokenizer = getattr(model_loader, 'translation_tokenizer', None)
        self.translation_available = getattr(model_loader, 'translation_available', False)
        self.nllb_model = getattr(model_loader, 'nllb_model', True)

        self.summarizer = getattr(model_loader, 'summarizer', None)
        self.summarization_available = getattr(model_loader, 'summarization_available', False)

        self.ner_pipeline = getattr(model_loader, 'ner_pipeline', None)
        self.ner_available = getattr(model_loader, 'ner_available', False)

        self.device = model_loader.device
        print(f"Assessment instance ready! Using device: {self.device}")

    def translate_text(self, text: str, source_lang: str = "kor_Hang", target_lang: str = "eng_Latn") -> Optional[str]:
        """Translate text using NLLB or Helsinki-NLP models"""
        if not self.translation_available:
            return None

        try:
            if not self.nllb_model:
                # Using Helsinki-NLP model
                inputs = self.translation_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = self.translation_model.generate(
                        **inputs,
                        max_length=512,
                        num_beams=4,
                        length_penalty=0.6,
                        early_stopping=True
                    )

                translation = self.translation_tokenizer.decode(outputs[0], skip_special_tokens=True)
                return translation
            else:
                # Using NLLB model
                self.translation_tokenizer.src_lang = source_lang
                inputs = self.translation_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = self.translation_model.generate(
                        **inputs,
                        forced_bos_token_id=self.translation_tokenizer.lang_code_to_id[target_lang],
                        max_length=512,
                        num_beams=4,
                        length_penalty=0.6,
                        early_stopping=True
                    )

                translation = self.translation_tokenizer.decode(outputs[0], skip_special_tokens=True)
                return translation

        except Exception as e:
            print(f"Translation error: {e}")
            return None

    def summarize_text(self, text: str, max_length: int = 150, min_length: int = 30) -> Optional[str]:
        """Summarize text using BART or T5"""
        if not self.summarization_available:
            return None

        try:
            if len(text.split()) < min_length:
                return "Text is too short to summarize effectively."

            summary = self.summarizer(
                text,
                max_length=max_length,
                min_length=min_length,
                do_sample=False
            )

            return summary[0]['summary_text'] if summary else None

        except Exception as e:
            print(f"Summarization error: {e}")
            return None

    def extract_entities(self, text: str) -> Optional[List[Dict]]:
        """Extract named entities from text"""
        if not self.ner_available:
            return None

        try:
            entities = self.ner_pipeline(text)

            formatted_entities = []
            for entity in entities:
                formatted_entities.append({
                    "text": entity.get("word", ""),
                    "label": entity.get("entity_group", entity.get("entity", "")),
                    "confidence": round(entity.get("score", 0.0), 3),
                    "start": entity.get("start", 0),
                    "end": entity.get("end", 0)
                })

            return formatted_entities

        except Exception as e:
            print(f"Entity extraction error: {e}")
            return None

    def get_text_statistics(self, text: str) -> Dict:
        """Get basic statistics about the text"""
        clean_text = re.sub(r'[^\w가-힣\s]', ' ', text)
        words = clean_text.split()

        korean_chars = len(re.findall(r'[가-힣]', text))
        english_chars = len(re.findall(r'[a-zA-Z]', text))

        return {
            "total_characters": len(text),
            "total_words": len(words),
            "korean_characters": korean_chars,
            "english_characters": english_chars,
            "sentences": len(re.split(r'[.!?]', text)) - 1,
            "average_word_length": sum(len(word) for word in words) / len(words) if words else 0
        }

    def analyze_text_content(self, text: str) -> Dict:
        """Comprehensive text analysis"""
        analysis_results = {
            "original_text": text,
            "translation": None,
            "summary": None,
            "entities": None,
            "text_stats": self.get_text_statistics(text)
        }

        # Translation
        if self.translation_available:
            translation = self.translate_text(text)
            analysis_results["translation"] = translation

            if translation:
                analysis_results["translated_entities"] = self.extract_entities(translation)
                analysis_results["translated_summary"] = self.summarize_text(translation)

        # Summarization
        if self.summarization_available:
            summary = self.summarize_text(text)
            if not summary and analysis_results["translation"]:
                summary = self.summarize_text(analysis_results["translation"])
            analysis_results["summary"] = summary

        # Entity extraction
        if self.ner_available:
            entities = self.extract_entities(text)
            analysis_results["entities"] = entities

        return analysis_results

In [ ]:
class KoreanPronunciationAssessor(KoreanPronunciationAssessor):

    def get_full_sentence_transcription(self, audio_path: str) -> Tuple[str, float]:
        """Get full sentence transcription with confidence using both models"""

        # Load audio
        audio, sr = librosa.load(audio_path, sr=16000)

        # Get Whisper transcription
        whisper_text = ""
        whisper_confidence = 0.5

        try:
            whisper_inputs = self.whisper_processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt"
            )

            # Handle different input feature names
            input_features = None
            for key in ['input_features', 'input_values']:
                if key in whisper_inputs:
                    input_features = whisper_inputs[key]
                    break

            if input_features is None:
                for key, value in whisper_inputs.items():
                    if isinstance(value, torch.Tensor):
                        input_features = value
                        break

            if input_features is None:
                raise ValueError("Could not find input features in Whisper processor output")

            input_features = input_features.to(self.device)

            forced_decoder_ids = self.whisper_processor.get_decoder_prompt_ids(
                language="korean",
                task="transcribe"
            )

            with torch.no_grad():
                whisper_outputs = self.whisper_model.generate(
                    input_features,
                    forced_decoder_ids=forced_decoder_ids,
                    max_length=200,
                    num_beams=5,
                    temperature=0.0,
                    do_sample=False
                )

            whisper_text = self.whisper_processor.batch_decode(
                whisper_outputs,
                skip_special_tokens=True
            )[0].strip()

            whisper_confidence = 0.9

        except Exception as e:
            print(f"Whisper processing error: {e}")
            whisper_text = ""
            whisper_confidence = 0.0

        # Get Wav2Vec2 transcription for validation
        wav2vec_text = ""

        try:
            wav2vec_inputs = self.wav2vec_processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt",
                padding=True
            )

            input_values = None
            for key in ['input_values', 'input_features']:
                if key in wav2vec_inputs:
                    input_values = wav2vec_inputs[key]
                    break

            if input_values is None:
                for key, value in wav2vec_inputs.items():
                    if isinstance(value, torch.Tensor):
                        input_values = value
                        break

            if input_values is not None:
                input_values = input_values.to(self.device)

                with torch.no_grad():
                    wav2vec_logits = self.wav2vec_model(input_values).logits
                    wav2vec_ids = torch.argmax(wav2vec_logits, dim=-1)
                    wav2vec_text = self.wav2vec_processor.batch_decode(wav2vec_ids)[0]

                wav2vec_text = re.sub(r'\s+', ' ', wav2vec_text).strip()

        except Exception as e:
            print(f"Wav2Vec2 processing error: {e}")
            wav2vec_text = ""

        print(f"Whisper: {whisper_text}")
        print(f"Wav2Vec2: {wav2vec_text}")

        # Use the better transcription
        if whisper_text and len(whisper_text) > len(wav2vec_text):
            return whisper_text, whisper_confidence
        elif wav2vec_text:
            return wav2vec_text, 0.8
        else:
            return whisper_text, whisper_confidence

    def calculate_levenshtein_similarity(self, str1: str, str2: str) -> float:
        """Calculate similarity using Levenshtein distance"""
        if not str1 or not str2:
            return 0.0

        def levenshtein_distance(s1, s2):
            if len(s1) < len(s2):
                return levenshtein_distance(s2, s1)

            if len(s2) == 0:
                return len(s1)

            previous_row = list(range(len(s2) + 1))
            for i, c1 in enumerate(s1):
                current_row = [i + 1]
                for j, c2 in enumerate(s2):
                    insertions = previous_row[j + 1] + 1
                    deletions = current_row[j] + 1
                    substitutions = previous_row[j] + (c1 != c2)
                    current_row.append(min(insertions, deletions, substitutions))
                previous_row = current_row

            return previous_row[-1]

        distance = levenshtein_distance(str1, str2)
        max_len = max(len(str1), len(str2))
        similarity = 1 - (distance / max_len) if max_len > 0 else 0

        return max(0, similarity)

    def calculate_word_score_and_status(self, expected: str, transcribed: str, confidence: float, similarity: float) -> Tuple[str, int]:
        """Calculate word score and status with updated categories"""

        expected_clean = re.sub(r'[^\w가-힣]', '', expected).strip()
        transcribed_clean = re.sub(r'[^\w가-힣]', '', transcribed).strip()

        if not transcribed_clean:
            return 'incorrect', 15

        if expected_clean == transcribed_clean:
            if confidence > 0.9:
                return 'correct', 95
            elif confidence > 0.8:
                return 'correct', 90
            elif confidence > 0.7:
                return 'correct', 85
            else:
                return 'try_harder', 80
        else:
            base_score = similarity * 100
            confidence_bonus = confidence * 10

            final_score = min(100, base_score + confidence_bonus)

            if final_score >= 85:
                return 'correct', int(final_score)
            elif final_score >= 70:
                return 'try_harder', int(final_score)
            elif final_score >= 50:
                return 'needs_improvement', int(final_score)
            else:
                return 'incorrect', int(final_score)

In [ ]:
class KoreanPronunciationAssessor(KoreanPronunciationAssessor):

    def preprocess_compound_words(self, expected_words: List[str], transcribed_words: List[str]) -> List[str]:
        """Handle compound words that might be split during transcription"""

        reconstructed_transcribed = []
        i = 0

        for expected_word in expected_words:
            if i >= len(transcribed_words):
                reconstructed_transcribed.append("")
                continue

            if len(expected_word) > 3:
                combined = ""
                original_i = i
                max_combine = min(3, len(transcribed_words) - i)

                for j in range(max_combine):
                    if i + j < len(transcribed_words):
                        test_combined = combined + transcribed_words[i + j]
                        similarity = self.calculate_levenshtein_similarity(expected_word, test_combined)

                        if similarity > 0.7:
                            combined = test_combined
                            i += j + 1
                            break
                        elif j == 0:
                            combined = test_combined
                else:
                    i = original_i + 1

                reconstructed_transcribed.append(combined)
            else:
                reconstructed_transcribed.append(transcribed_words[i] if i < len(transcribed_words) else "")
                i += 1

        return reconstructed_transcribed

    def align_words_with_transcription(self, expected_words: List[str], transcribed_words: List[str], confidence: float) -> List[Dict]:
        """Align expected words with transcribed words"""

        processed_transcribed = self.preprocess_compound_words(expected_words, transcribed_words)
        word_results = []

        for i, expected_word in enumerate(expected_words):
            transcribed_word = processed_transcribed[i] if i < len(processed_transcribed) else ""

            similarity = self.calculate_levenshtein_similarity(expected_word, transcribed_word)
            status, score = self.calculate_word_score_and_status(expected_word, transcribed_word, confidence, similarity)

            word_results.append({
                "word": expected_word,
                "status": status,
                "score": score,
                "transcribed": transcribed_word
            })

        return word_results

    def create_highlighted_text(self, word_results: List[Dict]) -> str:
        """Create highlighted text by grouping consecutive words with same status"""

        if not word_results:
            return ""

        highlighted_parts = []
        current_group = []
        current_status = word_results[0]['status']

        for word_data in word_results:
            word = word_data['word']
            status = word_data['status']

            if status == current_status:
                current_group.append(word)
            else:
                if current_group:
                    text = ' '.join(current_group)
                    highlighted_parts.append(f"<span class='{current_status}'>{text}</span>")

                current_group = [word]
                current_status = status

        if current_group:
            text = ' '.join(current_group)
            highlighted_parts.append(f"<span class='{current_status}'>{text}</span>")

        return ' '.join(highlighted_parts)

    def generate_specific_feedback(self, expected_words: List[str], transcribed_words: List[str], word_results: List[Dict]) -> List[str]:
        """Generate specific pronunciation feedback"""

        feedback = []
        processed_transcribed = self.preprocess_compound_words(expected_words, transcribed_words)

        for i, word_data in enumerate(word_results):
            expected = word_data['word']
            status = word_data['status']
            transcribed = processed_transcribed[i] if i < len(processed_transcribed) else ""

            if status == 'incorrect':
                if not transcribed:
                    feedback.append(f"'{expected}' 단어가 명확하게 들리지 않았어요. 더 크고 분명하게 발음해보세요.")
                else:
                    feedback.append(f"'{expected}'이 '{transcribed}'로 들렸어요. 정확한 발음을 연습해보세요.")

            elif status == 'needs_improvement':
                feedback.append(f"'{expected}' 발음에 더 신경써주세요. 좀 더 분발이 필요해요!")

            elif status == 'try_harder':
                if len(expected) > 0 and len(transcribed) > 0:
                    if '받침' in expected or any(char in expected for char in ['ㄱ', 'ㄴ', 'ㄷ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅅ']):
                        feedback.append(f"'{expected}'의 받침 발음이 약하게 들렸어요. 조금만 더 정확하게 발음해보세요.")
                    else:
                        feedback.append(f"'{expected}' 거의 정확해요! 조금만 더 열심히 연습하면 완벽할 거예요.")

        if not feedback:
            correct_count = sum(1 for w in word_results if w['status'] == 'correct')
            if correct_count == len(word_results):
                feedback.append("완벽한 발음입니다! 👏")
            else:
                feedback.append("전반적으로 좋은 발음이에요. 계속 연습하세요!")

        return feedback[:3]

    def calculate_pronunciation_score(self, word_results: List[Dict]) -> int:
        """Calculate overall pronunciation score"""

        if not word_results:
            return 0

        score_mapping = {
            'correct': 100,
            'try_harder': 75,
            'needs_improvement': 50,
            'incorrect': 25
        }

        total_score = sum(score_mapping.get(word['status'], 25) for word in word_results)
        average_score = total_score / len(word_results)

        return int(average_score)

    def get_color_legend(self) -> Dict[str, str]:
        """Return color legend mapping for frontend use"""
        return {
            'correct': {
                'color': 'green',
                'korean': '초록',
                'description': '완벽한 발음'
            },
            'try_harder': {
                'color': 'yellow',
                'korean': '노랑',
                'description': '조금만 더 열심히'
            },
            'needs_improvement': {
                'color': 'orange',
                'korean': '주황',
                'description': '분발 사항'
            },
            'incorrect': {
                'color': 'red',
                'korean': '빨강',
                'description': '오류'
            }
        }

In [ ]:
class KoreanPronunciationAssessor(KoreanPronunciationAssessor):

    def assess_sentence_pronunciation(self, audio_path: str, text: str, include_analysis: bool = True) -> Dict:
        """Main assessment function"""
        try:
            print("🎯 Starting pronunciation assessment...")

            # Get transcription
            full_transcription, full_confidence = self.get_full_sentence_transcription(audio_path)

            # Clean and prepare text
            expected_clean = re.sub(r'[^\w가-힣\s]', '', text).strip()
            transcribed_clean = re.sub(r'[^\w가-힣\s]', '', full_transcription).strip()

            expected_words = expected_clean.split()
            transcribed_words = transcribed_clean.split()

            print(f"📝 Expected: {expected_words}")
            print(f"🎤 Heard: {transcribed_words}")

            # Align words and assess
            word_results = self.align_words_with_transcription(expected_words, transcribed_words, full_confidence)

            # Calculate metrics
            pronunciation_score = self.calculate_pronunciation_score(word_results)
            highlighted_text = self.create_highlighted_text(word_results)
            processed_transcribed = self.preprocess_compound_words(expected_words, transcribed_words)
            feedback = self.generate_specific_feedback(expected_words, processed_transcribed, word_results)

            print("✅ Assessment completed!")

            # Print word-by-word results
            for word in word_results:
                status_emoji = {'correct': '✅', 'try_harder': '⚡', 'needs_improvement': '🔸', 'incorrect': '❌'}
                emoji = status_emoji.get(word['status'], '❓')
                print(f"{emoji} '{word['word']}': {word['status']} (Score: {word.get('score', 'N/A')})")

            # Base result
            result = {
                "sentence": text,
                "transcription": full_transcription,
                "words": word_results,
                "pronunciation_score": pronunciation_score,
                "highlighted_text": highlighted_text,
                "feedback": feedback,
                "color_legend": self.get_color_legend()
            }

            # Add text analysis if requested
            if include_analysis:
                print("📊 Performing text analysis...")
                text_analysis = self.analyze_text_content(text)
                result["text_analysis"] = text_analysis

                if full_transcription and full_transcription != text:
                    transcription_analysis = self.analyze_text_content(full_transcription)
                    result["transcription_analysis"] = transcription_analysis

            return result

        except Exception as e:
            print(f"❌ Error in assessment: {e}")
            import traceback
            traceback.print_exc()

            words = text.split()
            return {
                "sentence": text,
                "transcription": "",
                "words": [{"word": word, "status": "incorrect", "score": 25, "transcribed": ""} for word in words],
                "pronunciation_score": 30,
                "highlighted_text": f"<span class='incorrect'>{text}</span>",
                "feedback": [f"처리 중 오류가 발생했습니다: {str(e)}"],
                "text_analysis": None,
                "color_legend": self.get_color_legend()
            }

In [ ]:
from google.colab import files
import os
from IPython.display import Audio, display, HTML

class SimpleAudioUploader:
    def __init__(self):
        self.current_audio = None

    def upload_file(self):
        """Upload audio file / 오디오 파일 업로드"""
        print("📁 Upload WAV file / WAV 파일을 업로드하세요")

        uploaded = files.upload()

        if uploaded:
            filename = list(uploaded.keys())[0]
            self.current_audio = filename
            print(f"✅ Uploaded / 업로드 완료: {filename}")

            # Show audio player / 오디오 플레이어 표시
            display(Audio(filename))
            return filename
        else:
            print("❌ No file uploaded / 파일이 업로드되지 않음")
            return None

# Initialize uploader
uploader = SimpleAudioUploader()

In [ ]:
# Initialize the assessor with pre-loaded models
assessor = KoreanPronunciationAssessor(model_loader)

# Display color legend
print("\n" + "="*50)
print("🎨 COLOR LEGEND / 색상 범례")
print("="*50)
legend = assessor.get_color_legend()
for status, info in legend.items():
    print(f"{info['korean']} ({info['color']}): {info['description']}")
print("="*50)

Assessment instance ready! Using device: cuda

🎨 COLOR LEGEND / 색상 범례
초록 (green): 완벽한 발음
노랑 (yellow): 조금만 더 열심히
주황 (orange): 분발 사항
빨강 (red): 오류


In [ ]:
# from IPython.display import HTML, display

# # Test Case 1
# print("\n🧪 TEST CASE 1")
# print("="*30)
# audio_path = "Dialogue1.wav"
# text = "어떤 영화를 제일 좋아해요 저는 액션 영화를 좋아해요. 너무 재밌어요 저도요. 특히 마동석 배우가 나오는 영화요"

# result = assessor.assess_sentence_pronunciation(audio_path, text, include_analysis=True)
# print(f"\n📊 Overall Score: {result['pronunciation_score']}/100")
# print(f"🎯 Feedback: {result['feedback']}")

# # Display colored HTML
# css_style = """
# <style>
# .correct {
#     background-color: #90EE90;
#     color: #006400;
#     padding: 2px 4px;
#     border-radius: 3px;
#     font-weight: bold;
# }
# .try_harder {
#     background-color: #FFFF99;
#     color: #DAA520;
#     padding: 2px 4px;
#     border-radius: 3px;
#     font-weight: bold;
# }
# .needs_improvement {
#     background-color: #FFB366;
#     color: #FF8C00;
#     padding: 2px 4px;
#     border-radius: 3px;
#     font-weight: bold;
# }
# .incorrect {
#     background-color: #FFB6C1;
#     color: #DC143C;
#     padding: 2px 4px;
#     border-radius: 3px;
#     font-weight: bold;
# }
# .pronunciation-result {
#     font-size: 18px;
#     line-height: 1.6;
#     font-family: 'Malgun Gothic', Arial, sans-serif;
#     margin: 10px 0;
#     padding: 15px;
#     border: 2px solid #ddd;
#     border-radius: 8px;
#     background-color: #f9f9f9;
# }
# </style>
# """

# html_content = f"""
# {css_style}
# <div class="pronunciation-result">
#     <h3>🎯 발음 평가 결과 (Pronunciation Assessment Result)</h3>
#     <p><strong>원본 텍스트:</strong> {text}</p>
#     <p><strong>인식된 텍스트:</strong> {result['transcription']}</p>
#     <p><strong>색상별 평가:</strong></p>
#     <div style="font-size: 20px; margin: 15px 0;">
#         {result['highlighted_text']}
#     </div>
#     <p><strong>점수:</strong> {result['pronunciation_score']}/100</p>
# </div>
# """

# display(HTML(html_content))


🧪 TEST CASE 1
🎯 Starting pronunciation assessment...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Whisper: 어떤 영화를 제일 좋아해요? 저는 액션 영화를 좋아해요. 너무 재밌어요. 저도요. 특히 마동석 배우가 나오는 영화요.
Wav2Vec2: 어떤 영아을 제일 초화해어 등 액체 용어를 좋어해 노무조미스도 특히 마은속 배우가 나오경하요
📝 Expected: ['어떤', '영화를', '제일', '좋아해요', '저는', '액션', '영화를', '좋아해요', '너무', '재밌어요', '저도요', '특히', '마동석', '배우가', '나오는', '영화요']
🎤 Heard: ['어떤', '영화를', '제일', '좋아해요', '저는', '액션', '영화를', '좋아해요', '너무', '재밌어요', '저도요', '특히', '마동석', '배우가', '나오는', '영화요']
✅ Assessment completed!
✅ '어떤': correct (Score: 90)
✅ '영화를': correct (Score: 90)
✅ '제일': correct (Score: 90)
✅ '좋아해요': correct (Score: 90)
✅ '저는': correct (Score: 90)
✅ '액션': correct (Score: 90)
✅ '영화를': correct (Score: 90)
✅ '좋아해요': correct (Score: 90)
✅ '너무': correct (Score: 90)
✅ '재밌어요': correct (Score: 90)
✅ '저도요': correct (Score: 90)
✅ '특히': correct (Score: 90)
✅ '마동석': correct (Score: 90)
✅ '배우가': correct (Score: 90)
✅ '나오는': correct (Score: 90)
✅ '영화요': correct (Score: 90)
📊 Performing text analysis...
Translation error: NllbTokenizerFast has no attribute lang_code_to_id
Translation error: NllbTokenizerFast 

In [ ]:



# print("\n🧪 TEST CASE 2")
# print("="*30)
# audio_path = "/content/T0307G0151S0018.wav"
# text = "김화영이 번역하고 책세상에서 출간된 카뮈의 전집"

# result = assessor.assess_sentence_pronunciation(audio_path, text, include_analysis=True)
# print(f"\n📊 Overall Score: {result['pronunciation_score']}/100")
# print(f"🎯 Feedback: {result['feedback']}")

# # Display colored HTML
# css_style = """
# <style>
# .correct {
#     background-color: #90EE90;
#     color: #006400;
#     padding: 2px 4px;
#     border-radius: 3px;
#     font-weight: bold;
# }
# .try_harder {
#     background-color: #FFFF99;
#     color: #DAA520;
#     padding: 2px 4px;
#     border-radius: 3px;
#     font-weight: bold;
# }
# .needs_improvement {
#     background-color: #FFB366;
#     color: #FF8C00;
#     padding: 2px 4px;
#     border-radius: 3px;
#     font-weight: bold;
# }
# .incorrect {
#     background-color: #FFB6C1;
#     color: #DC143C;
#     padding: 2px 4px;
#     border-radius: 3px;
#     font-weight: bold;
# }
# .pronunciation-result {
#     font-size: 18px;
#     line-height: 1.6;
#     font-family: 'Malgun Gothic', Arial, sans-serif;
#     margin: 10px 0;
#     padding: 15px;
#     border: 2px solid #ddd;
#     border-radius: 8px;
#     background-color: #f9f9f9;
# }
# </style>
# """

# html_content = f"""
# {css_style}
# <div class="pronunciation-result">
#     <h3>🎯 발음 평가 결과 (Pronunciation Assessment Result)</h3>
#     <p><strong>원본 텍스트:</strong> {text}</p>
#     <p><strong>인식된 텍스트:</strong> {result['transcription']}</p>
#     <p><strong>색상별 평가:</strong></p>
#     <div style="font-size: 20px; margin: 15px 0;">
#         {result['highlighted_text']}
#     </div>
#     <p><strong>점수:</strong> {result['pronunciation_score']}/100</p>
# </div>
# """

# display(HTML(html_content))


🧪 TEST CASE 2
🎯 Starting pronunciation assessment...
Whisper: 김화영이 번역하고 책 세상에서 출간된 카뮤의 전집.
Wav2Vec2: 김화형이 번역하고 택 시상에서 출간된 카유에 전집
📝 Expected: ['김화영이', '번역하고', '책세상에서', '출간된', '카뮈의', '전집']
🎤 Heard: ['김화영이', '번역하고', '책', '세상에서', '출간된', '카뮤의', '전집']
✅ Assessment completed!
✅ '김화영이': correct (Score: 90)
✅ '번역하고': correct (Score: 90)
✅ '책세상에서': correct (Score: 90)
✅ '출간된': correct (Score: 90)
⚡ '카뮈의': try_harder (Score: 75)
✅ '전집': correct (Score: 90)
📊 Performing text analysis...
Translation error: NllbTokenizerFast has no attribute lang_code_to_id
Translation error: NllbTokenizerFast has no attribute lang_code_to_id

📊 Overall Score: 95/100
🎯 Feedback: ["'카뮈의' 거의 정확해요! 조금만 더 열심히 연습하면 완벽할 거예요."]


In [ ]:
def simple_test():
    """Simple pronunciation test / 간단한 발음 테스트"""

    print("🎤 Korean Pronunciation Test / 한국어 발음 테스트")
    print("="*50)

    # Step 1: Upload file / 1단계: 파일 업로드
    audio_path = uploader.upload_file()
    if not audio_path:
        return

    # Step 2: Enter text / 2단계: 텍스트 입력
    print("\n📝 Enter Korean text / 한국어 텍스트를 입력하세요:")
    text = input("Text / 텍스트: ").strip()

    if not text:
        print("❌ Please enter text / 텍스트를 입력해주세요")
        return

    # Step 3: Run test / 3단계: 테스트 실행
    print(f"\n🔄 Testing... / 테스트 중...")

    try:
        result = assessor.assess_sentence_pronunciation(audio_path, text, include_analysis=False)

        # Show results / 결과 표시
        print(f"\n📊 Score / 점수: {result['pronunciation_score']}/100")
        print(f"🎤 Recognized / 인식됨: {result['transcription']}")

        # Word analysis / 단어별 분석
        print(f"\n📝 Word Analysis / 단어별 분석:")
        for word_data in result['words']:
            emoji = {'correct': '✅', 'try_harder': '⚡', 'needs_improvement': '🔸', 'incorrect': '❌'}
            print(f"{emoji.get(word_data['status'], '❓')} {word_data['word']}: {word_data['status']}")

        # Show colored results / 색상 결과 표시
        show_colored_results(result, text)

    except Exception as e:
        print(f"❌ Error / 오류: {e}")

def show_colored_results(result, text):
    """Display colored results / 색상 결과 표시"""

    css = """
    <style>
    .correct { background-color: #90EE90; color: #006400; padding: 4px 8px; border-radius: 4px; font-weight: bold; }
    .try_harder { background-color: #FFFF99; color: #B8860B; padding: 4px 8px; border-radius: 4px; font-weight: bold; }
    .needs_improvement { background-color: #FFB366; color: #FF6600; padding: 4px 8px; border-radius: 4px; font-weight: bold; }
    .incorrect { background-color: #FFB6C1; color: #DC143C; padding: 4px 8px; border-radius: 4px; font-weight: bold; }
    .result-box { font-family: 'Malgun Gothic', Arial; padding: 20px; border: 2px solid #ddd; border-radius: 10px; background: #f9f9f9; margin: 20px 0; }
    .text-large { font-size: 22px; line-height: 1.8; margin: 15px 0; padding: 15px; background: white; border-radius: 8px; }
    </style>
    """

    html = f"""
    {css}
    <div class="result-box">
        <h3>🎯 Results / 결과</h3>

        <p><strong>Original / 원본:</strong> {text}</p>
        <p><strong>Recognized / 인식:</strong> {result['transcription']}</p>

        <p><strong>Color Analysis / 색상 분석:</strong></p>
        <div class="text-large">{result['highlighted_text']}</div>

        <p><strong>Score / 점수:</strong> {result['pronunciation_score']}/100</p>

        <p><strong>Legend / 범례:</strong>
        <span class="correct">Green/초록 = Perfect/완벽</span> |
        <span class="try_harder">Yellow/노랑 = Try harder/조금더</span> |
        <span class="needs_improvement">Orange/주황 = Needs work/분발</span> |
        <span class="incorrect">Red/빨강 = Error/오류</span>
        </p>
    </div>
    """

    display(HTML(html))

print("✅ Ready! Run simple_test() / 준비완료! simple_test() 실행하세요")

✅ Ready! Run simple_test() / 준비완료! simple_test() 실행하세요


In [ ]:
def quick_upload():
    """Just upload file / 파일만 업로드"""
    return uploader.upload_file()

def quick_test_with_text(text):
    """Quick test with provided text / 제공된 텍스트로 빠른 테스트"""
    if not uploader.current_audio:
        print("❌ Upload file first / 먼저 파일을 업로드하세요")
        return

    print(f"🔄 Testing: {text}")

    try:
        result = assessor.assess_sentence_pronunciation(uploader.current_audio, text)
        print(f"📊 Score: {result['pronunciation_score']}/100")
        show_colored_results(result, text)
    except Exception as e:
        print(f"❌ Error: {e}")

print("📌 Usage / 사용법:")
print("1. simple_test() - Full test / 전체 테스트")
print("2. quick_upload() - Upload only / 업로드만")
print("3. quick_test_with_text('your text') - Test with text / 텍스트로 테스트")

📌 Usage / 사용법:
1. simple_test() - Full test / 전체 테스트
2. quick_upload() - Upload only / 업로드만
3. quick_test_with_text('your text') - Test with text / 텍스트로 테스트


In [ ]:

simple_test()

🎤 Korean Pronunciation Test / 한국어 발음 테스트
📁 Upload WAV file / WAV 파일을 업로드하세요


Saving Dialogue1.wav to Dialogue1 (1).wav
✅ Uploaded / 업로드 완료: Dialogue1 (1).wav



📝 Enter Korean text / 한국어 텍스트를 입력하세요:
Text / 텍스트: 어떤 영화를 제일 좋아해요 저는 액션 영화를 좋아해요. 너무 재밌어요 저도요. 특히 마동석 배우가 나오는 영화요

🔄 Testing... / 테스트 중...
🎯 Starting pronunciation assessment...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Whisper: 어떤 영화를 제일 좋아해요? 저는 액션 영화를 좋아해요. 너무 재밌어요. 저도요. 특히 마동석 배우가 나오는 영화요.
Wav2Vec2: 어떤 영아을 제일 초화해어 등 액체 용어를 좋어해 노무조미스도 특히 마은속 배우가 나오경하요
📝 Expected: ['어떤', '영화를', '제일', '좋아해요', '저는', '액션', '영화를', '좋아해요', '너무', '재밌어요', '저도요', '특히', '마동석', '배우가', '나오는', '영화요']
🎤 Heard: ['어떤', '영화를', '제일', '좋아해요', '저는', '액션', '영화를', '좋아해요', '너무', '재밌어요', '저도요', '특히', '마동석', '배우가', '나오는', '영화요']
✅ Assessment completed!
✅ '어떤': correct (Score: 90)
✅ '영화를': correct (Score: 90)
✅ '제일': correct (Score: 90)
✅ '좋아해요': correct (Score: 90)
✅ '저는': correct (Score: 90)
✅ '액션': correct (Score: 90)
✅ '영화를': correct (Score: 90)
✅ '좋아해요': correct (Score: 90)
✅ '너무': correct (Score: 90)
✅ '재밌어요': correct (Score: 90)
✅ '저도요': correct (Score: 90)
✅ '특히': correct (Score: 90)
✅ '마동석': correct (Score: 90)
✅ '배우가': correct (Score: 90)
✅ '나오는': correct (Score: 90)
✅ '영화요': correct (Score: 90)

📊 Score / 점수: 100/100
🎤 Recognized / 인식됨: 어떤 영화를 제일 좋아해요? 저는 액션 영화를 좋아해요. 너무 재밌어요. 저도요. 특히 마동석 배우가 나오는 영화요.

📝 Word Analysis / 단어별 분석:
✅